# Publish deployment-ready checkpoints to the Hugging Face Hub

Two repos, one subfolder per checkpoint (an HF repo root can only hold one model's files -
`config.json`/`model.safetensors`/tokenizer files would collide if multiple checkpoints
were flattened into the same root, so each checkpoint gets its own `path_in_repo`
subfolder instead, mirroring the local `deployment/{architecture}/{run_id}/` layout
exactly).

- **`REPO_OTHER_LANGS`**: the 4 Kiswahili/Somali/Dholuo checkpoints (`06`'s combined model
  and all three independent per-language models).
- **`REPO_EKEGUSII`**: the 8 Ekegusii NLLB ablation configs (`02`) plus the mT5 PSA run
  (`04`) - explicitly excluding `nllb_zeroshot` (just the untouched base checkpoint, not a
  real deliverable) and `nllb_somali_target` (belongs in the other-languages repo, not
  here - it only sits in the same local `deployment/nllb/` folder because both
  architectures' checkpoints share one directory tree).

**Repos default to private** (`PRIVATE = True` below) - flip it to `False` once you've
confirmed you want these public. Set `HF_USERNAME` before running anything else.

## 1. Setup

In [1]:
!pip install -q huggingface_hub

In [2]:
import sys, os
from pathlib import Path

def find_project_root(marker="arch_config.py"):
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} by walking up from {current}")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"project root: {PROJECT_ROOT}")

from arch_config import DEPLOYMENT_DIR
DEPLOYMENT_DIR_PATH = Path(DEPLOYMENT_DIR)
print(f"deployment dir: {DEPLOYMENT_DIR_PATH.resolve()}")

project root: /home/jovyan/PSA/Final_Training
deployment dir: /home/jovyan/PSA/Final_Training/deployment


## 2. Authenticate

Needs a token with **Write** access from https://huggingface.co/settings/tokens - this
opens an interactive prompt/widget rather than taking the token as a plain argument here,
so it never ends up sitting in cell output or version control.

In [11]:
from huggingface_hub import login
login()

## 3. Define what goes where

Edit `HF_USERNAME` (and the repo names / `PRIVATE`, if you want something different) -
everything else derives from these.

In [4]:
HF_USERNAME = "SnitTeshome"   # <-- set this before running anything below
PRIVATE = True              # flip to False once you want these public

REPO_OTHER_LANGS = f"{HF_USERNAME}/nllb-kiswahili-somali-luo"
REPO_EKEGUSII = f"{HF_USERNAME}/nllb-ekegusii"

# (local path under deployment/, target repo, subfolder name within that repo)
UPLOAD_PLAN = [
    ("nllb/nllb_combined_other_langs", REPO_OTHER_LANGS, "nllb_combined_other_langs"),
    ("nllb/nllb_kiswahili_target",     REPO_OTHER_LANGS, "nllb_kiswahili_target"),
    ("nllb/nllb_somali_target",        REPO_OTHER_LANGS, "nllb_somali_target"),
    ("nllb/nllb_dholuo_target",        REPO_OTHER_LANGS, "nllb_dholuo_target"),

    ("nllb/nllb_mixed_lora_emb",       REPO_EKEGUSII, "nllb_mixed_lora_emb"),
    ("nllb/nllb_mixed_lora_only",      REPO_EKEGUSII, "nllb_mixed_lora_only"),
    ("nllb/nllb_psaonly_lora_emb",     REPO_EKEGUSII, "nllb_psaonly_lora_emb"),
    ("nllb/nllb_psaonly_lora_only",    REPO_EKEGUSII, "nllb_psaonly_lora_only"),
    ("nllb/nllb_stage1_lora_emb",      REPO_EKEGUSII, "nllb_stage1_lora_emb"),
    ("nllb/nllb_stage1_lora_only",     REPO_EKEGUSII, "nllb_stage1_lora_only"),
    ("nllb/nllb_stage2_lora_emb",      REPO_EKEGUSII, "nllb_stage2_lora_emb"),
    ("nllb/nllb_stage2_lora_only",     REPO_EKEGUSII, "nllb_stage2_lora_only"),
    ("mt5/mt5_mixed",                  REPO_EKEGUSII, "mt5_mixed"),
]

if HF_USERNAME == "CHANGE_ME":
    raise ValueError("Set HF_USERNAME above before continuing.")
print(f"{len(UPLOAD_PLAN)} checkpoints planned across 2 repos:")
print(f"  {REPO_OTHER_LANGS}: {sum(1 for _, r, _ in UPLOAD_PLAN if r == REPO_OTHER_LANGS)} checkpoints")
print(f"  {REPO_EKEGUSII}: {sum(1 for _, r, _ in UPLOAD_PLAN if r == REPO_EKEGUSII)} checkpoints")

13 checkpoints planned across 2 repos:
  SnitTeshome/nllb-kiswahili-somali-luo: 4 checkpoints
  SnitTeshome/nllb-ekegusii: 9 checkpoints


## 4. Verify every source folder is actually deployment-ready before uploading anything

Same check `inference.save_for_deployment()` already made when these were first saved -
re-confirmed here so a missing or corrupted folder fails loudly before any upload starts,
not partway through.

In [5]:
missing = []
for local_rel, repo_id, subfolder in UPLOAD_PLAN:
    folder = DEPLOYMENT_DIR_PATH / local_rel
    if not folder.exists():
        missing.append(f"{folder} - folder does not exist")
        continue
    if not list(folder.glob("*.safetensors")):
        missing.append(f"{folder} - no .safetensors file found")

if missing:
    raise FileNotFoundError("Fix these before uploading:\n" + "\n".join(missing))
print(f"all {len(UPLOAD_PLAN)} source folders verified present with a .safetensors file.")

all 13 source folders verified present with a .safetensors file.


## 5. Create both repos

In [6]:
from huggingface_hub import HfApi

api = HfApi()
for repo_id in dict.fromkeys(r for _, r, _ in UPLOAD_PLAN):   # de-duplicated, order preserved
    api.create_repo(repo_id, private=PRIVATE, exist_ok=True)
    print(f"repo ready: https://huggingface.co/{repo_id}  (private={PRIVATE})")

repo ready: https://huggingface.co/SnitTeshome/nllb-kiswahili-somali-luo  (private=True)
repo ready: https://huggingface.co/SnitTeshome/nllb-ekegusii  (private=True)


## 6. Upload each checkpoint into its own subfolder

In [7]:
for local_rel, repo_id, subfolder in UPLOAD_PLAN:
    folder = DEPLOYMENT_DIR_PATH / local_rel
    print(f"uploading {folder} -> {repo_id}/{subfolder} ...")
    api.upload_folder(folder_path=str(folder), repo_id=repo_id, path_in_repo=subfolder)
print("\nall uploads complete.")

uploading deployment/nllb/nllb_combined_other_langs -> SnitTeshome/nllb-kiswahili-somali-luo/nllb_combined_other_langs ...
uploading deployment/nllb/nllb_kiswahili_target -> SnitTeshome/nllb-kiswahili-somali-luo/nllb_kiswahili_target ...
uploading deployment/nllb/nllb_somali_target -> SnitTeshome/nllb-kiswahili-somali-luo/nllb_somali_target ...
uploading deployment/nllb/nllb_dholuo_target -> SnitTeshome/nllb-kiswahili-somali-luo/nllb_dholuo_target ...
uploading deployment/nllb/nllb_mixed_lora_emb -> SnitTeshome/nllb-ekegusii/nllb_mixed_lora_emb ...
uploading deployment/nllb/nllb_mixed_lora_only -> SnitTeshome/nllb-ekegusii/nllb_mixed_lora_only ...
uploading deployment/nllb/nllb_psaonly_lora_emb -> SnitTeshome/nllb-ekegusii/nllb_psaonly_lora_emb ...
uploading deployment/nllb/nllb_psaonly_lora_only -> SnitTeshome/nllb-ekegusii/nllb_psaonly_lora_only ...
uploading deployment/nllb/nllb_stage1_lora_emb -> SnitTeshome/nllb-ekegusii/nllb_stage1_lora_emb ...
uploading deployment/nllb/nllb_stag

In [20]:
!pip install -q huggingface_hub

import sys, os
from pathlib import Path

def find_project_root(marker="arch_config.py"):
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} by walking up from {current}")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from arch_config import CHECKPOINTS_DIR, DEPLOYMENT_DIR
import inference
from huggingface_hub import login, HfApi

login()   # skip if you're already authenticated in this kernel session

HF_USERNAME = "SnitTeshome"   # <-- set this
REPO_OTHER_LANGS = f"{HF_USERNAME}/nllb-kiswahili-somali-luo"
RUN_ID = "nllb_combined_other_langs_v3"

merged_path = Path(CHECKPOINTS_DIR) / RUN_ID / "merged"
if not merged_path.exists():
    raise FileNotFoundError(f"{merged_path} not found - has 06_nllb_other_languages_v3.ipynb "
                            f"finished training and saved a merged checkpoint?")
inference.save_for_deployment(str(merged_path), "nllb", RUN_ID, deployment_dir=DEPLOYMENT_DIR)

deploy_path = Path(DEPLOYMENT_DIR) / "nllb" / RUN_ID
if not list(deploy_path.glob("*.safetensors")):
    raise FileNotFoundError(f"no .safetensors file in {deploy_path} - don't upload this")

api = HfApi()
api.create_repo(REPO_OTHER_LANGS, private=True, exist_ok=True)
print(f"uploading {deploy_path} -> {REPO_OTHER_LANGS}/{RUN_ID} ...")
api.upload_folder(folder_path=str(deploy_path), repo_id=REPO_OTHER_LANGS, path_in_repo=RUN_ID)
print("upload complete.")

existing_files = api.list_repo_files(REPO_OTHER_LANGS)
subfolders = sorted({f.split("/")[0] for f in existing_files if "/" in f})
readme = "\n".join([
    f"# {REPO_OTHER_LANGS}", "",
    "Multiple checkpoints in one repo, one subfolder per checkpoint. Load a specific "
    "one with the `subfolder` argument:", "",
    "```python",
    "from transformers import AutoModelForSeq2SeqLM, AutoTokenizer", "",
    f'model = AutoModelForSeq2SeqLM.from_pretrained("{REPO_OTHER_LANGS}", subfolder="<name>")',
    f'tokenizer = AutoTokenizer.from_pretrained("{REPO_OTHER_LANGS}", subfolder="<name>")',
    "```", "", "Available subfolders:", "",
] + [f"- `{s}`" for s in subfolders])
api.upload_file(path_or_fileobj=readme.encode("utf-8"), path_in_repo="README.md", repo_id=REPO_OTHER_LANGS)
print(f"README.md updated - subfolders now: {subfolders}")

deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_combined_other_langs_v3  (1 safetensors file(s))
uploading deployment/nllb/nllb_combined_other_langs_v3 -> SnitTeshome/nllb-kiswahili-somali-luo/nllb_combined_other_langs_v3 ...
upload complete.
README.md updated - subfolders now: ['nllb_combined_other_langs', 'nllb_combined_other_langs_v2', 'nllb_combined_other_langs_v3', 'nllb_dholuo_target', 'nllb_kiswahili_target', 'nllb_somali_target']


In [21]:
HF_USERNAME = "SnitTeshome"   # <-- set this before running anything below
PRIVATE = True              # flip to False once you want these public

REPO_OTHER_LANGS = f"{HF_USERNAME}/nllb-kiswahili-somali-luo"
REPO_EKEGUSII = f"{HF_USERNAME}/nllb-ekegusii"

# (local path under deployment/, target repo, subfolder name within that repo)
UPLOAD_PLAN = [
    ("nllb/nllb_combined_other_langs",    REPO_OTHER_LANGS, "nllb_combined_other_langs"),
    ("nllb/nllb_combined_other_langs_v2", REPO_OTHER_LANGS, "nllb_combined_other_langs_v2"),
    ("nllb/nllb_combined_other_langs_v3", REPO_OTHER_LANGS, "nllb_combined_other_langs_v3"),
    ("nllb/nllb_kiswahili_target",     REPO_OTHER_LANGS, "nllb_kiswahili_target"),
    ("nllb/nllb_somali_target",        REPO_OTHER_LANGS, "nllb_somali_target"),
    ("nllb/nllb_dholuo_target",        REPO_OTHER_LANGS, "nllb_dholuo_target"),

    ("nllb/nllb_mixed_lora_emb",       REPO_EKEGUSII, "nllb_mixed_lora_emb"),
    ("nllb/nllb_mixed_lora_only",      REPO_EKEGUSII, "nllb_mixed_lora_only"),
    ("nllb/nllb_psaonly_lora_emb",     REPO_EKEGUSII, "nllb_psaonly_lora_emb"),
    ("nllb/nllb_psaonly_lora_only",    REPO_EKEGUSII, "nllb_psaonly_lora_only"),
    ("nllb/nllb_stage1_lora_emb",      REPO_EKEGUSII, "nllb_stage1_lora_emb"),
    ("nllb/nllb_stage1_lora_only",     REPO_EKEGUSII, "nllb_stage1_lora_only"),
    ("nllb/nllb_stage2_lora_emb",      REPO_EKEGUSII, "nllb_stage2_lora_emb"),
    ("nllb/nllb_stage2_lora_only",     REPO_EKEGUSII, "nllb_stage2_lora_only"),
    ("mt5/mt5_mixed",                  REPO_EKEGUSII, "mt5_mixed"),
]

if HF_USERNAME == "CHANGE_ME":
    raise ValueError("Set HF_USERNAME above before continuing.")
print(f"{len(UPLOAD_PLAN)} checkpoints planned across 2 repos:")
print(f"  {REPO_OTHER_LANGS}: {sum(1 for _, r, _ in UPLOAD_PLAN if r == REPO_OTHER_LANGS)} checkpoints")
print(f"  {REPO_EKEGUSII}: {sum(1 for _, r, _ in UPLOAD_PLAN if r == REPO_EKEGUSII)} checkpoints")

15 checkpoints planned across 2 repos:
  SnitTeshome/nllb-kiswahili-somali-luo: 6 checkpoints
  SnitTeshome/nllb-ekegusii: 9 checkpoints


## 7. Write a root README per repo

Without this, a repo holding several checkpoints under subfolders gives no hint of what's
there or how to load a specific one.

In [22]:
def build_readme(repo_id):
    subfolders = [s for _, r, s in UPLOAD_PLAN if r == repo_id]
    lines = [
        f"# {repo_id}",
        "",
        "Multiple checkpoints in one repo, one subfolder per checkpoint. Load a specific "
        "one with the `subfolder` argument:",
        "",
        "```python",
        "from transformers import AutoModelForSeq2SeqLM, AutoTokenizer",
        "",
        f'model = AutoModelForSeq2SeqLM.from_pretrained("{repo_id}", subfolder="<name>")',
        f'tokenizer = AutoTokenizer.from_pretrained("{repo_id}", subfolder="<name>")',
        "```",
        "",
        "Available subfolders:",
        "",
    ]
    lines += [f"- `{s}`" for s in subfolders]
    return "\n".join(lines)

for repo_id in dict.fromkeys(r for _, r, _ in UPLOAD_PLAN):
    api.upload_file(
        path_or_fileobj=build_readme(repo_id).encode("utf-8"),
        path_in_repo="README.md",
        repo_id=repo_id,
    )
    print(f"README.md written to {repo_id}")

No files have been modified since last commit. Skipping to prevent empty commit.


README.md written to SnitTeshome/nllb-kiswahili-somali-luo
README.md written to SnitTeshome/nllb-ekegusii


## 8. Verify: list what's actually on the Hub now

In [23]:
for repo_id in dict.fromkeys(r for _, r, _ in UPLOAD_PLAN):
    files = api.list_repo_files(repo_id)
    subfolders = sorted({f.split("/")[0] for f in files if "/" in f})
    expected = sorted({s for _, r, s in UPLOAD_PLAN if r == repo_id})

    missing = sorted(set(expected) - set(subfolders))
    extra = sorted(set(subfolders) - set(expected))

    if missing:
        status = f"MISSING {missing} - check manually"
    elif extra:
        status = f"OK"
    else:
        status = "OK"

    print(f"{repo_id}: {len(subfolders)} subfolders on the Hub -> {subfolders}  [{status}]")

SnitTeshome/nllb-kiswahili-somali-luo: 6 subfolders on the Hub -> ['nllb_combined_other_langs', 'nllb_combined_other_langs_v2', 'nllb_combined_other_langs_v3', 'nllb_dholuo_target', 'nllb_kiswahili_target', 'nllb_somali_target']  [OK]
SnitTeshome/nllb-ekegusii: 9 subfolders on the Hub -> ['mt5_mixed', 'nllb_mixed_lora_emb', 'nllb_mixed_lora_only', 'nllb_psaonly_lora_emb', 'nllb_psaonly_lora_only', 'nllb_stage1_lora_emb', 'nllb_stage1_lora_only', 'nllb_stage2_lora_emb', 'nllb_stage2_lora_only']  [OK]
